In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os, sys

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

from superstore_platform_config import get_catalog, get_mart_schema, get_semantic_layer_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
semantic_layer_schema = get_semantic_layer_schema()
mart_schema = get_mart_schema()  

from superstore_logger import get_superstore_logger, log_event
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_metrics_daily_kpi= get_superstore_logger("superstore_metrics_daily_kpi")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_metrics_daily_kpi,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="kpi view"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
required_tables = [
    f"{catalog}.{mart_schema}.mart_sales_daily"
]
missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, mart_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_metrics_daily_kpi, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create metrics daily kpi view.", layer="kpi view")
else:
  # -------------------------------
  # Create kpi View (Semantic Layer for Analysts)
  # -------------------------------
  spark.sql(f"""
  CREATE OR REPLACE VIEW {catalog}.{semantic_layer_schema}.metrics_daily_kpi AS
  SELECT
    order_date,
    SUM(total_sales) AS total_sales,
    SUM(total_profit) AS total_profit,
    ROUND(SUM(total_profit)/SUM(total_sales),4) AS profit_margin
  FROM {catalog}.{mart_schema}.mart_sales_daily
  GROUP BY order_date
  ORDER BY order_date;
  """)
log_event(logger_metrics_daily_kpi, "INFO", "Metrics daily kpi view created successfully.", layer="kpi view")
